시작 전 Anthropic API, OpenAI API 발급 받아야 합니다.
- Anthropic API : https://console.anthropic.com/settings/plans
- OpenAI API : https://platform.openai.com/settings/profile

In [ ]:
!pip install anthropic pymupdf openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 891.9/891.9 kB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.9/15.9 MB 64.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 367.8/367.8 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.9/77.9 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.9/318.9 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 4.7 MB/s eta 0:00:00


In [ ]:
import os
import re
import numpy as np
import base64
import requests
from tqdm import tqdm
import anthropic
from PIL import Image
import fitz
import io
import pandas as pd
import random
import ast
from openai import OpenAI
import json
import fitz
import xml.etree.ElementTree as ET
import xml.dom.minidom as minidom

## Key

In [ ]:
# API 키를 환경 변수에 설정
os.environ['ANTHROPIC_API_KEY'] =  'ANTHROPIC_API_KEY'
client = OpenAI(api_key="OPENAI_API_KEY")

## 1. PDF 페이지를 이미지로 저장

In [ ]:
# 이미지를 base64 형식으로 인코딩하는 함수
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode('utf-8')

# 이미지가 다단인지 확인하는 함수

    # 이미지가 컬러라면 RGB를 그레이스케일로 변환
    if len(img_array.shape) == 3:
        center_region = np.mean(center_region, axis=2)

    vertical_projection = np.mean(center_region, axis=1)
    white_ratio = np.mean(vertical_projection > threshold)

    return white_ratio > 0.9  # 중심 부분이 90% 이상 흰색일 때 다단으로 간주

In [ ]:
# PNG 이미지를 최적화하는 함수
def optimize_png(img):
    img_buffer = io.BytesIO()
    img.save(img_buffer, format='PNG', optimize=True, compress_level=6)
    optimized_img = Image.open(img_buffer)
    return optimized_img

# PDF 파일을 처리하여 각 페이지를 이미지로 저장하는 함수
def process_pdf(pdf_path, output_folder, dpi=300):
    doc = fitz.open(pdf_path)
    os.makedirs(output_folder, exist_ok=True)

    for page_num in range(len(doc)):
        page = doc[page_num]
        zoom = dpi / 72
        mat = fitz.Matrix(zoom, zoom)
        pix = page.get_pixmap(matrix=mat, alpha=False)

        img = Image.frombytes("RGB", [pix.width, pix.height], pix.samples)
        img_array = np.array(img)

        # 이미지 최적화 후 저장
        optimize_png(img).save(os.path.join(output_folder, f'page_{page_num+1}.png'))

        '''
        # 다단 이미지일 경우, 좌/우측 이미지를 따로 저장
        if is_two_column(img_array):
            left_half = img.crop((0, 0, img.width//2, img.height))
            right_half = img.crop((img.width//2, 0, img.width, img.height))

            optimize_png(left_half).save(os.path.join(output_folder, f'page_{page_num+1}_left.png'))
            optimize_png(right_half).save(os.path.join(output_folder, f'page_{page_num+1}_right.png'))
        else:
            optimize_png(img).save(os.path.join(output_folder, f'page_{page_num+1}.png'))
        '''

    doc.close()

In [ ]:
# 사용 예시: PDF 파일을 이미지로 변환
pdf_path = '가입자교육.pdf'
output_folder = '.'
process_pdf(pdf_path, output_folder)

In [ ]:
files = os.listdir()

In [ ]:
files

['.config',
 'page_31.png',
 'page_5.png',
 'page_25.png',
 'page_3.png',
 '가입자교육.pdf',
 'page_22.png',
 'page_28.png',
 'page_13.png',
 'page_9.png',
 'page_32.png',
 'page_11.png',
 'page_24.png',
 'page_14.png',
 'page_20.png',
 'page_7.png',
 'page_19.png',
 'page_23.png',
 'page_2.png',
 'page_12.png',
 'page_29.png',
 'page_16.png',
 'page_33.png',
 'page_36.png',
 'page_26.png',
 'page_35.png',
 'page_1.png',
 'page_10.png',
 'page_21.png',
 'page_6.png',
 'page_15.png',
 'page_34.png',
 'page_17.png',
 'page_30.png',
 'page_27.png',
 'page_8.png',
 'page_4.png',
 'page_18.png',
 'sample_data']

## 2. PDF 페이지를 xml로 저장

In [ ]:
# 페이지에서 텍스트를 추출하여 XML로 변환하는 함수
def extract_page_content_to_xml(page):
    text = page.get_text("text")
    return text

# 텍스트에서 XML 구조를 생성하는 함수
def create_xml_structure_from_text(text):
    root = ET.Element("page")
    text_elem = ET.SubElement(root, "text")
    text_elem.text = text
    return root

# XML 콘텐츠를 정돈하여 출력 형식으로 변환하는 함수
def prettify_xml_content(elem):
    rough_string = ET.tostring(elem, 'utf-8')
    reparsed = minidom.parseString(rough_string)
    return reparsed.toprettyxml(indent="  ")

# PDF를 XML로 변환하여 저장하는 함수
def convert_pdf_to_xml(pdf_path, output_folder):
    if not os.path.exists(pdf_path):
        print(f"PDF 파일을 찾을 수 없습니다: {pdf_path}")
        return

    os.makedirs(output_folder, exist_ok=True)

    print(f"PDF 파일 열기: {pdf_path}")
    doc = fitz.open(pdf_path)

    for page_num in range(len(doc)):
        print(f"페이지 {page_num + 1}/{len(doc)} 처리 중...")
        page = doc[page_num]
        text = extract_page_content_to_xml(page)

        if not text.strip():
            print(f"경고: 페이지 {page_num + 1}에 추출할 텍스트가 없습니다.")
            continue

        root = create_xml_structure_from_text(text)
        try:
            xml_string = prettify_xml_content(root)

            output_file = os.path.join(output_folder, f"page_{page_num+1:02d}.xml")
            print(f"XML 파일 저장 중: {output_file}")
            with open(output_file, "w", encoding="utf-8") as f:
                f.write(xml_string)

            print(f"페이지 {page_num + 1} 처리 완료")
        except:
            output_file = os.path.join(output_folder, f"page_{page_num+1:02d}.xml")
            print(f"추출에 실패하여 값이 빈 XML 파일 저장 중: {output_file}")
            with open(output_file, "w", encoding="utf-8") as f:
                f.write('')

    doc.close()
    print("모든 페이지 처리 완료")

In [ ]:
pdf_path = '가입자교육.pdf'
output_folder = '.'
convert_pdf_to_xml(pdf_path, output_folder)

PDF 파일 열기: 가입자교육.pdf
페이지 1/36 처리 중...
XML 파일 저장 중: ./page_01.xml
페이지 1 처리 완료
페이지 2/36 처리 중...
XML 파일 저장 중: ./page_02.xml
페이지 2 처리 완료
페이지 3/36 처리 중...
XML 파일 저장 중: ./page_03.xml
페이지 3 처리 완료
페이지 4/36 처리 중...
XML 파일 저장 중: ./page_04.xml
페이지 4 처리 완료
페이지 5/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_05.xml
페이지 6/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_06.xml
페이지 7/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_07.xml
페이지 8/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_08.xml
페이지 9/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_09.xml
페이지 10/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_10.xml
페이지 11/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_11.xml
페이지 12/36 처리 중...
XML 파일 저장 중: ./page_12.xml
페이지 12 처리 완료
페이지 13/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_13.xml
페이지 14/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_14.xml
페이지 15/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_15.xml
페이지 16/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_16.xml
페이지 17/36 처리 중...
추출에 실패하여 값이 빈 XML 파일 저장 중: ./page_1

In [ ]:
# 디렉토리에서 매칭되는 XML과 PNG 파일들을 가져오는 함수
def get_matched_files(directory):
    files = os.listdir(directory)

    xml_files = {}
    png_files = {}

    # XML과 PNG 파일들을 분류
    for file in files:
        match = re.match(r'page_(\d+)\.(xml|png)$', file)
        if match:
            number = int(match.group(1))
            extension = match.group(2)
            if extension == 'xml':
                xml_files[number] = file
            elif extension == 'png':
                png_files[number] = file

    # 모든 페이지 번호를 결합하여 매칭
    all_numbers = set(xml_files.keys()).union(set(png_files.keys()))

    matched_files = []
    for number in sorted(all_numbers):
        xml_path = os.path.join(directory, xml_files[number]) if number in xml_files else 'no xml'
        png_path = os.path.join(directory, png_files[number]) if number in png_files else 'no png'
        matched_files.append((xml_path, png_path))

    return matched_files

In [ ]:
%pwd

'/content'

## 3. 각 페이지로부터 이미지와 XML 매칭

In [ ]:
# 사용 예시: 매칭된 파일 가져오기
directory = r'/content'
files = get_matched_files(directory)

In [ ]:
files

[('/content/page_01.xml', '/content/page_1.png'),
 ('/content/page_02.xml', '/content/page_2.png'),
 ('/content/page_03.xml', '/content/page_3.png'),
 ('/content/page_04.xml', '/content/page_4.png'),
 ('/content/page_05.xml', '/content/page_5.png'),
 ('/content/page_06.xml', '/content/page_6.png'),
 ('/content/page_07.xml', '/content/page_7.png'),
 ('/content/page_08.xml', '/content/page_8.png'),
 ('/content/page_09.xml', '/content/page_9.png'),
 ('/content/page_10.xml', '/content/page_10.png'),
 ('/content/page_11.xml', '/content/page_11.png'),
 ('/content/page_12.xml', '/content/page_12.png'),
 ('/content/page_13.xml', '/content/page_13.png'),
 ('/content/page_14.xml', '/content/page_14.png'),
 ('/content/page_15.xml', '/content/page_15.png'),
 ('/content/page_16.xml', '/content/page_16.png'),
 ('/content/page_17.xml', '/content/page_17.png'),
 ('/content/page_18.xml', '/content/page_18.png'),
 ('/content/page_19.xml', '/content/page_19.png'),
 ('/content/page_20.xml', '/content/page

In [ ]:
# XML이 없는 경우 사용할 시스템 프롬프트
no_xml_system_prompt = '''당신이 해석할 이미지는 교육 내용입니다.
1. 중요한 내용이므로 요약하지말고 문법에 신경쓰면서 보이는 그대로 작성해주세요.
2. 내용을 임의로 바꾸지 마세요. 그리고 보이는 모든 내용을 다 적으십시오.
3. 단, 테이블은 풀어서 평문 또는 나열식으로 작성해주세요. 이미지에 없는 말은 적지마세요.
4. 테이블 풀어서 평문 또는 나열식으로 작성할 때 다른 행과 열이랑 헷갈리지 않게 값마다 잘 구분해서 적어주세요.
5. 테이블 해석할 때 통합셀들이 존재하니 구조를 잘 해석해서 작성해주시기 바랍니다. 어떤 게 어떤 것의 하위 내용인지를 명확히 하십시오
6. 당신의 의견은 궁금하지 않습니다. 해드렸습니다. 완성했습니다. 이런 표현도 적지마십시오. 이미지에 있는 내용만 적으십시오.
7. 만약 다단으로 구성되어져 있다면 좌측 테이블부터 먼저 작성하고 우측 테이블을 작성하십시오.

자, 당신이 모든 내용을 빠트리지 않으면서 테이블은 구조를 잘 해석해서 작성해주는 것을 믿습니다.
'''

In [ ]:
# XML이 있는 경우 사용할 시스템 프롬프트
system_prompt = '''당신이 해석할 이미지는 교육 내용입니다.
1. 중요한 내용이므로 요약하지말고 문법에 신경쓰면서 보이는 그대로 작성해주세요.
2. 내용을 임의로 바꾸지 마세요. 그리고 보이는 모든 내용을 다 적으십시오.
3. 단, 테이블은 풀어서 평문 또는 나열식으로 작성해주세요. 이미지에 없는 말은 적지마세요.
4. 테이블 풀어서 평문 또는 나열식으로 작성할 때 다른 행과 열이랑 헷갈리지 않게 값마다 잘 구분해서 적어주세요.
5. 테이블 해석할 때 통합셀들이 존재하니 구조를 잘 해석해서 작성해주시기 바랍니다. 어떤 게 어떤 것의 하위 내용인지를 명확히 하십시오
6. 당신의 의견은 궁금하지 않습니다. 해드렸습니다. 완성했습니다. 이런 표현도 적지마십시오. 이미지에 있는 내용만 적으십시오.
7. 만약 다단으로 구성되어져 있다면 좌측 테이블부터 먼저 작성하고 우측 테이블을 작성하십시오.
8. 당신에게 당신이 해석할 파일을 xml로 변경한 내용도 드리겠습니다. 페이지 해석할 때 참고하세요.
9. xml에 있는 텍스트는 반드시 해당 페이지에 존재한다는 겁니다. xml에 있는 텍스트를 빠트리지 마십시오.


자 당신이 헷갈리지 않도록 xml도 드렸습니다. 이미지를 더 잘 해석할 거라 믿습니다.
'''

In [ ]:
# Anthropic API 클라이언트 초기화
client = anthropic.Anthropic()

In [ ]:
# 이미지와 XML 파일을 사용하여 AI 모델에 요청을 보내는 코드
result_lst = []
for file in tqdm(files):
    image_path = file[1]

    # 이미지를 base64 형식으로 인코딩
    base64_image = encode_image(image_path)

    if file[0] == 'no xml':
        print(file[1], '은 xml이 없습니다.')
        prompt = no_xml_system_prompt
    else:
        with open(file[0], 'r', encoding='utf-8') as f:
            xml_content = f.read()
        prompt = system_prompt + xml_content + '\n시작!'

    # AI 모델에 메시지 생성 요청
    message = client.messages.create(
        model="claude-3-5-sonnet-20240620",
        max_tokens=4096,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": "image/png",
                            "data": base64_image,
                        },
                    },
                    {
                        "type": "text",
                        "text": prompt
                    }
                ],
            }
        ],
    )
    result_lst.append(message.content[0].text)

100%|██████████| 36/36 [09:50<00:00, 16.41s/it]


In [ ]:
# 결과를 페이지와 연결하여 저장
result = []
for f, r in zip(files, result_lst):
    result.append({'content': r, 'source': 'page_' + f[1].split('page_')[1]})

In [ ]:
result

[{'page_content': "TONGYANG LIFE INSURANCE\n\n퇴직연금\n가입자\n교육자료\n\n이 이미지는 동양생명(TONGYANG LIFE INSURANCE)의 퇴직연금 가입자 교육자료 표지입니다. 파란색 그라데이션 배경에 흰색 글씨로 회사명과 자료 제목이 한글과 영어로 표시되어 있습니다.\n\n중앙에는 회사의 마스코트로 보이는 캐릭터가 있습니다. 이 캐릭터는 흰색 둥근 머리에 파란 하트 모양의 장식이 있고, 검은 양복을 입고 있으며, 한 손에 돋보기를 들고 있습니다. 캐릭터는 작은 날개도 가지고 있습니다.\n\n하단 오른쪽에는 회사 로고로 보이는 파란색 물결 모양의 디자인과 함께 '동양생명'이라는 한글 텍스트가 있습니다.\n\n전체적으로 이 표지는 보험회사의 퇴직연금 교육 자료임을 명확히 보여주는 디자인으로 구성되어 있습니다.",
  'source': 'page_1.png'},
 {'page_content': 'Contents\n\nⅠ 퇴직연금제도 일반  \nⅡ 확정급여형(DB)제도 추가 교육\nⅢ 확정기여형(DC) 및 개인형퇴직연금(IRP) 추가 교육\n\n※ 가입자 교육이란?\n근로자퇴직급여보장법 제32조 제2항의 규정에 따라 사용자로부터 교육의 실시를 위탁받은 사업자 또는 동법 제33조 제5항의 규정에 따라 개인형퇴직연금제도를 운영하는 사업자가 집합, 서면 또는 온라인 등에 의한 방법으로 가입자에게 매년 1회 이상 실시하는 교육을 말합니다.',
  'source': 'page_2.png'},
 {'page_content': 'PART. Ⅰ\n퇴직연금제도 일반', 'source': 'page_3.png'},
 {'page_content': 'PART.Ⅰ 퇴직연금제도 일반\n\n1. 급여종류에 관한 사항, 수급요건, 급여 등 제도별 특징 및 차이점\n\n[1] 퇴직연금제도란?\n기업이 사내에 적립하던 퇴직금제도를 대체하여, 매년 퇴직금 지급을 위한 재원을 외부 금융기관에 적립하여 근로자가 퇴직할 때 연금 또는 일시금으로 지급받는 안정

In [ ]:
result = result[4:]

In [ ]:
result

[{'content': "TONGYANG LIFE INSURANCE\n\n퇴직연금\n가입자\n교육자료\n\n이 이미지는 동양생명(TONGYANG LIFE INSURANCE)의 퇴직연금 가입자 교육자료 표지입니다. 파란색 그라데이션 배경에 흰색 글씨로 회사명과 자료 제목이 한글과 영어로 표시되어 있습니다.\n\n중앙에는 회사의 마스코트로 보이는 캐릭터가 있습니다. 이 캐릭터는 흰색 둥근 머리에 파란 하트 모양의 장식이 있고, 검은 양복을 입고 있으며, 한 손에 돋보기를 들고 있습니다. 캐릭터는 작은 날개도 가지고 있습니다.\n\n하단 오른쪽에는 회사 로고로 보이는 파란색 물결 모양의 디자인과 함께 '동양생명'이라는 한글 텍스트가 있습니다.\n\n전체적으로 이 표지는 보험회사의 퇴직연금 교육 자료임을 명확히 보여주는 디자인으로 구성되어 있습니다.",
  'source': 'page_1.png'},
 {'content': 'Contents\n\nⅠ 퇴직연금제도 일반  \nⅡ 확정급여형(DB)제도 추가 교육\nⅢ 확정기여형(DC) 및 개인형퇴직연금(IRP) 추가 교육\n\n※ 가입자 교육이란?\n근로자퇴직급여보장법 제32조 제2항의 규정에 따라 사용자로부터 교육의 실시를 위탁받은 사업자 또는 동법 제33조 제5항의 규정에 따라 개인형퇴직연금제도를 운영하는 사업자가 집합, 서면 또는 온라인 등에 의한 방법으로 가입자에게 매년 1회 이상 실시하는 교육을 말합니다.',
  'source': 'page_2.png'},
 {'content': 'PART. Ⅰ\n퇴직연금제도 일반', 'source': 'page_3.png'},
 {'content': 'PART.Ⅰ 퇴직연금제도 일반\n\n1. 급여종류에 관한 사항, 수급요건, 급여 등 제도별 특징 및 차이점\n\n[1] 퇴직연금제도란?\n기업이 사내에 적립하던 퇴직금제도를 대체하여, 매년 퇴직금 지급을 위한 재원을 외부 금융기관에 적립하여 근로자가 퇴직할 때 연금 또는 일시금으로 지급받는 안정된 노후를 위한 노후설계가 가능하도록

In [ ]:
!pip install langchain langchain-openai tiktoken openai faiss-cpu pypdf chromadb langchain_community

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 6.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 55.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.0/27.0 MB 82.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 584.3/584.3 kB 43.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 86.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.8/273.8 kB 24.2 MB/s eta 0:00:

In [ ]:
import os
import re
import getpass
import matplotlib.pyplot as plt
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

In [ ]:
import os
OPENAI_API_KEY = "OPEN_API_KEY"

os.environ['OPENAI_API_KEY'] =  OPENAI_API_KEY

In [ ]:
# prompt: result 딕셔너리를 langchain Document 타입으로 변환

from langchain.docstore.document import Document

# result 딕셔너리를 langchain Document 타입으로 변환
documents = [Document(page_content=item['content'], metadata={"source": item['source']}) for item in result]

In [ ]:
embedding = OpenAIEmbeddings()

vectordb = Chroma.from_documents(
    documents=documents,
    embedding=embedding)

In [ ]:
retriever = vectordb.as_retriever()

In [ ]:
docs = retriever.get_relevant_documents("가입자 교육의 의미")
print('유사 문서 개수 :', len(docs))
print('--' * 20)
print('첫번째 유사 문서 :', docs[0])
print('--' * 20)
print('각 유사 문서의 문서 출처 :')
for doc in docs:
    print(doc.metadata["source"])

<ipython-input-42-bd984e6fa1df>:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use invoke instead.
  docs = retriever.get_relevant_documents("가입자 교육의 의미")


유사 문서 개수 : 4
----------------------------------------
첫번째 유사 문서 : page_content='Contents

Ⅰ 퇴직연금제도 일반  
Ⅱ 확정급여형(DB)제도 추가 교육
Ⅲ 확정기여형(DC) 및 개인형퇴직연금(IRP) 추가 교육

※ 가입자 교육이란?
근로자퇴직급여보장법 제32조 제2항의 규정에 따라 사용자로부터 교육의 실시를 위탁받은 사업자 또는 동법 제33조 제5항의 규정에 따라 개인형퇴직연금제도를 운영하는 사업자가 집합, 서면 또는 온라인 등에 의한 방법으로 가입자에게 매년 1회 이상 실시하는 교육을 말합니다.' metadata={'source': 'page_2.png'}
----------------------------------------
각 유사 문서의 문서 출처 :
page_2.png
page_22.png
page_28.png
page_1.png


In [ ]:
qa_chain = RetrievalQA.from_chain_type(
    llm=ChatOpenAI(model_name="gpt-4o-mini", temperature=0),
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True)

In [ ]:
input_text = "사용자 부담금이 궁금해요. 아주 간단하게 이야기 해주세요"
chatbot_response = qa_chain(input_text)
print(chatbot_response)

{'query': '사용자 부담금이 궁금해요. 아주 간단하게 이야기 해주세요', 'result': '사용자의 부담금은 근로자의 연간 임금총액의 1/12 이상을 퇴직연금 계좌에 납입해야 합니다. 추가로 근로자가 스스로 부담금을 더 납입할 수도 있습니다.', 'source_documents': [Document(metadata={'source': 'page_27.png'}, page_content='1. 사용자의 부담금 수준, 납입시기 및 납입현황\n\n[1] 가입자가 자신의 수령금액을 점검하고 부담금에 대한 관심을 가질수 있도록 부담금과 관련한 교육사항\n- 부담금 인기도에 전에 가입자에게 개별 안내합니다.\n- 부담금 이율변동 시 사전에 가입자에게 개별 안내합니다.\n- 부담금 납입현황 등 내역서를 연1회 이상 가입자에게 제공합니다.\n\n[2] 부담금 수준과 관련한 교육 사항\n① 가입자별 사용자의 부담금액 또는 그 금액을 확인할 수 있는 방법\n- 동양생명 홈페이지(www.myangel.co.kr) 통합로그인 → 퇴직연금 "온용현황" 메뉴에서 확인 가능합니다.\n② 부담금 산정 방법\n- DC를 설정한 사용자는 가입자의 연간 임금총액의 1/12분의1 이상에 해당하는 부담금을 현금으로 가입자의 DC 계정에 납입하여야 합니다.\n- 가입자는 사용자가 부담하는 부담금 외에 스스로 부담하는 추가 부담금을 가입자의 DC 계정에 납입할 수 있습니다.\n(연간 납입 한도 1800만원, 연금계좌 합산하여 연간 900만원 한도로 세액공제 가능)\n\n[3] 부담금 납입시기와 관련한 교육 사항\n① 부담금 납입시기 및 납입 기한\n- 확정기여형 퇴직연금 규약에 명시 된 납입 주기에 따라 매년 1회 이상 정기적으로 부담금을 납입하여야 합니다.\n(납입주기: 연납 / 반기납 / 분기납 / 월납 으로 선택 가능)\n② 부담금 납입기한이 경과한 경우 미납금액에 대한 처리방법\n- 부담금 미납금액 납입 전에 퇴직연금사업자에 확인하여 미납부담금 및 지연이자를 확인하고 납부하도록 합니다

In [ ]:
input_text = "퇴직연금이 무엇인가요?"
chatbot_response = qa_chain(input_text)
print(chatbot_response)

{'query': '퇴직연금이 무엇인가요?', 'result': '퇴직연금제도란 기업이 사내에 적립하던 퇴직금제도를 대체하여, 매년 퇴직금 지급을 위한 재원을 외부 금융기관에 적립하여 근로자가 퇴직할 때 연금 또는 일시금으로 지급받는 안정된 노후를 위한 노후설계가 가능하도록 한 선진형 기업복지제도입니다. 이 제도는 2005년 12월 1일 근로자퇴직급여보장법의 시행에 따라 제도화되었습니다.', 'source_documents': [Document(metadata={'source': 'page_4.png'}, page_content='PART.Ⅰ 퇴직연금제도 일반\n\n1. 급여종류에 관한 사항, 수급요건, 급여 등 제도별 특징 및 차이점\n\n[1] 퇴직연금제도란?\n기업이 사내에 적립하던 퇴직금제도를 대체하여, 매년 퇴직금 지급을 위한 재원을 외부 금융기관에 적립하여 근로자가 퇴직할 때 연금 또는 일시금으로 지급받는 안정된 노후를 위한 노후설계가 가능하도록 한 선진형 기업복지제도입니다. (2005년 12월 1일 근로자퇴직급여보장법의 시행에 따라 제도화 되었습니다.)\n\n[2] 퇴직연금제도의 종류\n퇴직급여제도는 퇴직연금제도와 퇴직(일시)금제도로 나뉩니다. 퇴직연금제도는 다시 확정급여형(DB)과 확정기여형(DC)으로 구분됩니다. 또한 개인형퇴직연금제도(IRP)와 IRP특례(기업형IRP)가 있습니다.\n\n1) 확정급여형(DB, Defined Benefit)제도\n- 근로자의 퇴직급여가 근무기간과 평균임금에 의하여 확정되며, 사용자가 적립금을 운용하는 제도입니다.\n\n급여종류 / 수급요건: 연금은 10년 이상 가입 / 55세 이상 / 5년 이상 연금 수령기간 지정이 필요합니다. 일시금은 연금수급 자격을 갖추지 못했거나 일시금 수령을 원하는 자가 받을 수 있습니다.\n\n급여수준: 근속연수 1년에 대해 30일분의 평균임금 (기존의 퇴직금과 동일함)\n\n이미지에는 사용자, 금융기관, 근로자의 관계를 나타내는 그림이 포함되어 있습니다. 사용자가 운용하고, 금융기관

In [ ]:
def get_chatbot_response(chatbot_response):
    print(chatbot_response['result'].strip())
    print('\n문서 출처:')
    for source in chatbot_response["source_documents"]:
        print(source.metadata['source'])

In [ ]:
input_text = "가입자교육의 목적"
chatbot_response = qa_chain(input_text)
get_chatbot_response(chatbot_response)

가입자 교육의 목적은 근로자퇴직급여보장법에 따라 가입자에게 퇴직연금 제도에 대한 이해를 돕고, 적립금의 안정적 운용 및 투자 원칙에 대한 교육을 제공하여 가입자가 자신의 자산을 적정하게 운영할 수 있도록 하는 것입니다. 이를 통해 가입자는 퇴직연금 제도의 규정, 부담금 납입 현황, 투자 원칙 등을 이해하고, 자신의 투자 성향에 맞는 적절한 투자 결정을 내릴 수 있도록 지원받게 됩니다.

문서 출처:
page_2.png
page_22.png
page_28.png
page_26.png


In [ ]:
input_text = "퇴직 연금 제도는 어떤 사람들이 받기 좋아?"
llm_response = qa_chain(input_text)
get_chatbot_response(llm_response)

퇴직 연금 제도는 주로 다음과 같은 사람들에게 유리합니다:

1. **장기 근속자**: 퇴직 연금은 근무 기간에 따라 퇴직급여가 결정되므로, 오랜 기간 동안 한 회사에서 일한 사람들에게 유리합니다.

2. **안정적인 노후를 원하는 사람**: 퇴직 연금은 퇴직 후 안정적인 소득을 제공하므로, 노후에 경제적 안정을 원하는 사람들에게 적합합니다.

3. **연금 수령 요건을 충족하는 사람**: 만 55세 이상이고, 가입 후 5년이 경과한 사람은 연금을 수령할 수 있는 자격이 있으므로, 이러한 요건을 충족하는 사람들에게 유리합니다.

4. **퇴직금 대신 연금을 선호하는 사람**: 일시금 수령 대신 정기적으로 연금을 받고 싶어하는 사람들에게 적합합니다.

이러한 조건을 충족하는 사람들은 퇴직 연금 제도를 통해 더 나은 혜택을 받을 수 있습니다.

문서 출처:
page_4.png
page_14.png
page_13.png
page_27.png


In [ ]:
input_text = "적립금은 뭐야?"
llm_response = qa_chain(input_text)
get_chatbot_response(llm_response)

적립금은 퇴직연금 제도에서 근로자가 퇴직 후 받을 수 있는 금액을 의미합니다. 이는 사용자가 근로자의 퇴직급여를 지급하기 위해 적립하는 금액으로, 원리금 보장형 상품이나 실적 배당형 상품 등 다양한 형태로 운용될 수 있습니다. 적립금은 근로자의 퇴직급여 재원 안정성을 고려하여 신중하게 운용되며, 법정 최소 적립금을 항상 유지해야 합니다.

문서 출처:
page_25.png
page_24.png
page_27.png
page_9.png


In [ ]:
input_text = "적립금이 최소 적립금보다 적은 경우 어떻게 해야해?"
llm_response = qa_chain(input_text)
get_chatbot_response(llm_response)

적립금이 최소적립금의 95% 미만일 경우, 사용자는 부족금액을 최대 3년 이내에 균등하게 해소할 수 있도록 재정안정화계획서를 작성해야 합니다. 이 계획서에는 부족금액에 대한 자금조달방안, 납입계획 등의 내용이 포함되어야 하며, 사업자로부터 재정검증결과를 통보받은 날부터 60일 이내에 해당 사업자와 근로자의 과반수가 가입한 노동조합이 있는 경우에는 그 노동조합, 근로자의 과반수가 가입한 노동조합이 없는 경우에는 전체 근로자에게 재정안정화계획서를 통보해야 합니다. 또한, 사용자는 재정안정화계획서를 3년간 보존해야 합니다.

문서 출처:
page_24.png
page_25.png
page_27.png
page_16.png


In [ ]:
input_text = "동양생명 퇴직연금 디폴트옵션 상품 안내해줘. 승인 일자는 년-월-일 형식으로. 승인 년도가 2023년 이후인 것만"
llm_response = qa_chain(input_text)
get_chatbot_response(llm_response)

동양생명 퇴직연금 디폴트옵션 상품 중 승인 년도가 2023년 이후인 상품은 다음과 같습니다:

1. 상품명: 동양생명 디폴트옵션 저위험 PF2
   - 승인일자: 2023-03-07

2. 상품명: 동양생명 디폴트옵션 중위험 PF2
   - 승인일자: 2023-03-07

3. 상품명: 동양생명 디폴트옵션 고위험 BF2
   - 승인일자: 2023-03-07

문서 출처:
page_33.png
page_32.png
page_36.png
page_31.png


In [ ]:
input_text = "실적 배당형 상품의 매수와 환매 기준가가 시간에 따라 어떤 프로세스가 있는지 알려줘. 매수와 환매 모두"
llm_response = qa_chain(input_text)
get_chatbot_response(llm_response)

실적배당형 상품의 매수와 환매 기준가는 다음과 같은 프로세스를 따릅니다:

**매수 프로세스:**
1. [접수일 D]: 매수 접수
2. [D + 1 영업일]: 매수 운용지시
3. [D + 2 영업일]: 매수 정산(기준가격 매수)

**환매 프로세스:**
1. [접수일 D]: 매도 접수
2. [D + 1 영업일]: 매도 운용지시
3. [D + 4 영업일]: 매도 정산(기준가격 매도)

문서 출처:
page_31.png
page_34.png
page_13.png
page_27.png


In [ ]:
input_text = "퇴직연금제도에는 어떤것들이 있어?"
llm_response = qa_chain(input_text)
get_chatbot_response(llm_response)

퇴직연금제도는 크게 두 가지로 나뉘며, 각각의 세부 유형은 다음과 같습니다:

1. **퇴직연금제도**
   - **확정급여형(DB, Defined Benefit) 제도**: 근로자의 퇴직급여가 근무기간과 평균임금에 의해 확정되며, 사용자가 적립금을 운용하는 제도입니다.
   - **확정기여형(DC, Defined Contribution) 제도**: 사용자가 매년 일정 금액을 적립하고, 근로자가 퇴직 시 그 적립금을 기반으로 연금이나 일시금을 수령하는 제도입니다.
   - **개인형퇴직연금제도(IRP)**: 개인이 자발적으로 가입하여 퇴직금을 적립하는 제도입니다.
   - **IRP특례(기업형IRP)**: 기업이 운영하는 개인형퇴직연금제도입니다.

2. **퇴직(일시)금제도**: 퇴직 시 일시금으로 지급되는 제도입니다. 

이와 같이 퇴직연금제도는 여러 가지 유형으로 구성되어 있습니다.

문서 출처:
page_4.png
page_2.png
page_3.png
page_13.png


In [ ]:
input_text = "개인형 퇴직 연금제도의 이체를 위한 요건과 효과에 대해 아주 자세히 설명해줘"
llm_response = qa_chain(input_text)
get_chatbot_response(llm_response)

개인형 퇴직 연금제도의 이체를 위한 요건과 효과는 다음과 같습니다.

### 이체를 위한 요건
1. **가입자의 연령**: 가입자가 만 55세를 경과해야 합니다.
2. **계좌 가입일**: 가입일로부터 5년 이상 경과해야 합니다. 이 조건은 연금 수령 조건을 충족한 경우에 해당합니다.
3. **퇴직금 입금 여부**: 개인형IRP의 경우 퇴직금이 입금된 경우에는 5년 경과 규정이 제외됩니다. 즉, 퇴직금이 입금된 후에는 이체가 가능하다는 의미입니다.

### 이체에 따른 효과
1. **과세이연 효과**: 이체 시 인출로 보지 않기 때문에 과세되지 않고 이연되는 효과가 있습니다. 이는 가입자가 일시금으로 수령할 때까지 퇴직소득세 납부 시기를 이연함으로써 투자수익 증가 효과를 기대할 수 있습니다.
2. **다양한 상품군**: 개인형IRP로 이체함으로써 원리금 보장형 및 실적 배당형 상품 등 다양한 상품군을 통해 고객의 성향에 맞춘 포트폴리오 설계가 가능합니다.
3. **절세효과**: 가입자가 55세 이후 연금으로 수령할 경우, 연금소득세 자율고제로 납세액의 경감 효과를 누릴 수 있습니다.

이러한 요건과 효과를 통해 개인형 퇴직 연금제도의 이체는 가입자에게 유리한 조건을 제공하며, 재정적 안정성을 높이는 데 기여할 수 있습니다.

문서 출처:
page_4.png
page_10.png
page_35.png
page_16.png


In [ ]:
input_text = "연금저축 계좌 및 개인형 IRP 이체 절차를 설명해줘"
llm_response = qa_chain(input_text)
get_chatbot_response(llm_response)

연금저축 계좌 및 개인형 IRP 이체 절차는 다음과 같습니다:

1. **가입자 중심의 절차**:
   - 가입자는 신규 계좌를 개설합니다.
   - 가입자는 이체신청서를 작성하고, 계좌이체 시 유의사항을 확인한 후 서명합니다.
   - 가입자는 이체 결과를 확인하고 통보받습니다(녹취).

2. **이체 전 계좌의 금융기관과 이체받을 계좌의 금융기관 사이의 절차**:
   - 이체받을 계좌의 금융기관에서 이체 전 계좌의 금융기관으로 이체 예정 또는 취소 통보를 FAX 발송합니다.
   - 이체 전 계좌의 금융기관에서 이체받을 계좌의 금융기관으로 이체 접수 또는 거절 통보를 FAX 발송합니다.
   - 이체 전 계좌의 금융기관에서 금융상품을 매도한 후 송금하며, 연금계좌 이체 명세서 등을 발송합니다.

이체를 위한 요건으로는 가입자의 연령이 만 55세를 초과하고 계좌 가입일로부터 5년 이상 경과해야 하며, 연금수령 조건을 충족해야 합니다. 이체 시 과세되지 않고 이연되는 효과가 있습니다.

문서 출처:
page_35.png
page_10.png
page_15.png
page_11.png


In [ ]:
!pip install gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.1/18.1 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 318.7/318.7 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.9/93.9 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.3/10.3 MB 125.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 12.0 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 13.0.1
    Uninstalling websockets-13.0.1:
      Successfully uninstalled websockets-13.0.1
  Attempting uninstall: tomlkit
    Found existing installation: tomlkit 0.13.2
    Uninstalling tomlkit-0.13.2:
      Successfully uninstalled tomlkit-0.13.2
  Attempting uninstall: fastapi
    Found existing installation: fastapi 0.114.0
    Uninstalling fastapi-0.114.0:
      Successfully uninstalled fastapi-0.114.0


In [ ]:
import gradio as gr

# 인터페이스를 생성.
with gr.Blocks() as demo:
    chatbot = gr.Chatbot(label="동양생명 가입자 교육 챗봇")
    msg = gr.Textbox(label="질문해주세요!")  # 하단의 채팅창의 레이블
    clear = gr.Button("대화 초기화")  # 대화 초기화 버튼

    # 챗봇의 답변을 처리하는 함수
    def respond(message, chat_history):
      result = qa_chain(message)
      bot_message = result['result']
      bot_message += ' # sources :'

      # 답변의 출처를 표기
      for i, doc in enumerate(result['source_documents']):
          bot_message += '[' + str(i+1) + '] ' + doc.metadata['source'] + ' '

      # 채팅 기록에 사용자의 메시지와 봇의 응답을 추가.
      chat_history.append((message, bot_message))
      return "", chat_history

    # 사용자의 입력을 제출(submit)하면 respond 함수가 호출.
    msg.submit(respond, [msg, chatbot], [msg, chatbot])

    # '초기화' 버튼을 클릭하면 채팅 기록을 초기화.
    clear.click(lambda: None, None, chatbot, queue=False)

# 인터페이스 실행.
demo.launch(debug=True)

Setting queue=True in a Colab notebook requires sharing enabled. Setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
Running on public URL: https://47685c77d33d4a3aa1.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from Terminal to deploy to Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://47685c77d33d4a3aa1.gradio.live
